# SmartSeniors — Biblio Emma (version **Drive-persistante**)

Même pipeline (**appels → fiches → playbook**), mais **tout est stocké sur ton Google Drive**.
Tu déposes tes `.m4a` **une seule fois** dans un dossier Drive ; le notebook ne **re-transcrit / ne re-analyse que les nouveaux** appels, et le playbook couvre toujours **tout le corpus** — même après une réinitialisation de Colab.

**Dossier de travail (sur ton Drive)** : `MyDrive/SmartSeniors/biblio_emma/`
- `audio/` → tu y déposes tes `.m4a` (glisser-déposer dans Drive)
- `transcripts/` → transcripts bruts (persistés)
- `fiches/` → fiches conseillères anonymisées (`.json` + `.md`)
- `emma-playbook.md` → la synthèse finale

**Avant de lancer** : Exécution → Modifier le type d'exécution → **T4 GPU**.
**RGPD** : `audio/` et transcripts bruts restent sur **ton** Drive privé ; tu ne committes que l'**anonymisé** (`emma-playbook.md`, `fiches/`).

## 0 — Setup + montage du Drive (à lancer une fois par session)

In [ ]:
!nvidia-smi -L 2>/dev/null || echo "⚠️ Active un GPU : Exécution → Modifier le type d'exécution → T4 GPU, puis relance."
!pip -q install -U "faster-whisper>=1.1.0" anthropic

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE        = "/content/drive/MyDrive/SmartSeniors/biblio_emma"
AUDIO       = f"{BASE}/audio"
TRANSCRIPTS = f"{BASE}/transcripts"
FICHES      = f"{BASE}/fiches"
for d in (AUDIO, TRANSCRIPTS, FICHES):
    os.makedirs(d, exist_ok=True)
print("📁 Dossier de travail :", BASE)
print("   → dépose tes .m4a dans :", AUDIO)

from getpass import getpass
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Clé API Anthropic (sk-ant-…) : ")

import torch
from faster_whisper import WhisperModel
device = "cuda" if torch.cuda.is_available() else "cpu"
WHISPER = WhisperModel("large-v3", device=device, compute_type="float16" if device == "cuda" else "int8")
print("✅ Whisper large-v3 prêt sur", device)

## 1 — Déposer les appels & transcrire (idempotent)

Dépose tes `.m4a` (anciens **et** nouveaux) dans `audio/` sur ton Drive — une seule fois, par glisser-déposer dans Drive.
Tu peux aussi en **ajouter directement depuis Colab** ci-dessous (ils seront copiés dans `audio/`).
La cellule ne transcrit **que les appels pas encore transcrits** : relançable sans recalcul inutile.

In [ ]:
import glob, os, shutil

# (Optionnel) Ajouter des .m4a directement depuis Colab → copiés dans Drive/audio
try:
    from google.colab import files
    print("Optionnel : choisis des fichiers à AJOUTER (ou Annuler s'ils sont déjà dans Drive/audio)…")
    up = files.upload()
    for name in up:
        shutil.move(name, f"{AUDIO}/{os.path.basename(name)}")
        print("   ➕ ajouté :", os.path.basename(name))
except Exception as e:
    print("Upload ignoré :", e)

def hms(s):
    s = int(s); return f"{s//3600:02d}:{(s%3600)//60:02d}:{s%60:02d}"

audios = sorted(sum([glob.glob(f"{AUDIO}/*{ext}") for ext in (".m4a", ".mp3", ".wav", ".mp4", ".ogg")], []))
print(f"\n🎧 {len(audios)} fichier(s) audio dans Drive/audio")
for path in audios:
    nom = os.path.splitext(os.path.basename(path))[0]
    out = f"{TRANSCRIPTS}/{nom}.txt"
    if os.path.exists(out):
        print(f"   ⏭️  {nom} — déjà transcrit, skip")
        continue
    print(f"   🎙️  {nom} …")
    segs, info = WHISPER.transcribe(path, language="fr", vad_filter=True, beam_size=5)
    timed = "\n".join(f"[{hms(s.start)}] {s.text.strip()}" for s in segs)
    open(out, "w").write(timed)
    print(f"      {info.duration:.0f}s → {out}")

print("\n📄 transcripts :", [os.path.basename(p) for p in sorted(glob.glob(f"{TRANSCRIPTS}/*.txt"))])

## 2 — Schéma de la fiche + extracteur (Claude `opus-4-8`, tool use, anonymisé)

Sortie structurée fiable (tool use forcé + Pydantic). **Anonymisation** : pré-passe regex + prompt strict.
**Prompt caching** : le préfixe `system + outil` est mis en cache (`cache_read` > 0 dès le 2ᵉ appel).

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional, Literal

class QuestionDecouverte(BaseModel):
    ordre: int = Field(description="Position dans l'appel (1 = première question)")
    question: str = Field(description="La question telle que posée, reformulée proprement")
    intention: str = Field(description="Ce que le conseiller cherche à apprendre")
    etape: str = Field(description="contexte_emotionnel | profil | autonomie | localisation | budget | delai | criteres")
    champ_lead: Optional[str] = Field(default=None, description="Champ de lead renseigné, si applicable")

class PointDonnee(BaseModel):
    champ_lead: str = Field(description="Champ de lead, ex: prenom_proche, ville_recherche, budget_mensuel, delai")
    valeur: str = Field(description="Valeur captée (ANONYMISÉE si c'est une donnée personnelle)")
    verbatim: str = Field(description="Extrait du transcript (ANONYMISÉ) d'où vient l'info")
    etape: Literal["identite", "solution"] = Field(description="identite = création du lead ; solution = critères de matching")

class Objection(BaseModel):
    objection: str = Field(description="Ce que la famille exprime comme frein")
    reponse_conseiller: str = Field(description="Comment le conseiller y répond")
    technique: str = Field(description="Nom de la technique (réassurance financière, dédramatisation, etc.)")

class PhraseEmpathie(BaseModel):
    phrase: str
    contexte: str = Field(description="Quand cette phrase est employée")

class Transition(BaseModel):
    de: str
    vers: str
    formulation: str = Field(description="La phrase qui fait passer d'une étape à l'autre")

class FicheConseillere(BaseModel):
    resume_appel: str
    questions_decouverte: List[QuestionDecouverte]
    points_donnees: List[PointDonnee]
    objections: List[Objection]
    phrases_empathie: List[PhraseEmpathie]
    vocabulaire: List[str] = Field(description="Tournures et mots employés à réutiliser par Emma")
    transitions: List[Transition]
    bonnes_pratiques: List[str] = Field(description="Ce qui rend ce conseiller efficace")
    a_eviter: List[str] = Field(description="Maladresses à ne pas reproduire")

In [ ]:
import anthropic, json, re

client = anthropic.Anthropic()
MODEL = "claude-opus-4-8"          # qualité max ; pour de gros volumes, tu peux tester "claude-sonnet-4-6"

# Pré-anonymisation mécanique (filet de sécurité AVANT l'envoi à Claude)
def pre_anon(t: str) -> str:
    t = re.sub(r"(?:\+33|0)\s*[1-9](?:[ .\-]?\d{2}){4}", "[TEL]", t)            # téléphones FR
    t = re.sub(r"[\w.+-]+@[\w-]+\.[\w.-]+", "[EMAIL]", t)                       # emails
    t = re.sub(r"\b(?:[A-Za-zÀ-ÿ]-){2,}[A-Za-zÀ-ÿ]\b", "[NOM ÉPELÉ]", t)       # noms épelés "V-I-V-A-N-T"
    return t

SYSTEM_ANALYSTE = """Tu es analyste qualité chez SmartSeniors (mise en relation familles ↔ EHPAD).
À partir du transcript d'un appel (conseiller ↔ famille en recherche d'établissement), tu extrais une « fiche conseillère »
qui servira à former Emma, l'IA conseillère.

RÈGLES D'ANONYMISATION — STRICTES, AUCUNE EXCEPTION.
Dans TOUS les champs (résumé, verbatims, valeurs, objections…), remplace par [ANONYMISÉ] :
- le nom de famille ET le prénom de la famille comme du senior (même épelés lettre par lettre) ;
- la commune/ville précise de résidence de la famille, l'adresse, le téléphone, l'email ;
- le nom précis d'un établissement, d'une résidence ou d'une personne tierce nommée (« Mme X »).
Tu CONSERVES (utile au métier, non identifiant) : le département (ex: 91, 78), le type d'établissement,
les montants, les délais, le GIR, les aides, la situation (hôpital/SSR/domicile).
Pour un champ d'identité (prénom/nom/contact), mets la valeur « [ANONYMISÉ] » mais GARDE la ligne
(pour montrer que l'info a bien été captée pendant l'appel).

QUALITÉ :
- Reformule proprement (corrige les fautes du speech-to-text) SANS inventer.
- Mappe chaque donnée au champ de lead : prenom_proche, nom_proche, date_naissance_proche, ville_recherche,
  code_postal, rayon_km, niveau_autonomie, situation_actuelle, budget_mensuel, delai, lien_proche,
  type_residence, contact_prenom, contact_nom, contact_telephone, contact_email.
- Étape « identite » = infos pour CRÉER le lead (nom, prénom, date de naissance, CP + ville).
  Étape « solution » = critères de matching (délai, ville + rayon, budget).
- Pour budget_mensuel, distingue bien : revenus du senior, capacité contributive de la famille, et coût estimé."""

FICHE_TOOL = {
    "name": "enregistrer_fiche_conseillere",
    "description": "Enregistre la fiche conseillère structurée extraite de l'appel.",
    "input_schema": FicheConseillere.model_json_schema(),
}

def extraire_fiche(transcript: str) -> FicheConseillere:
    resp = client.messages.create(
        model=MODEL, max_tokens=16000,
        system=[{"type": "text", "text": SYSTEM_ANALYSTE, "cache_control": {"type": "ephemeral"}}],
        tools=[FICHE_TOOL],
        tool_choice={"type": "tool", "name": "enregistrer_fiche_conseillere"},
        messages=[{"role": "user", "content": [
            {"type": "text", "text": "TRANSCRIPT :\n\n" + pre_anon(transcript), "cache_control": {"type": "ephemeral"}},
            {"type": "text", "text": "Analyse ce transcript et remplis la fiche conseillère, en respectant les règles d'anonymisation."},
        ]}],
    )
    u = resp.usage
    print(f"   📊 in={u.input_tokens} cache_write={u.cache_creation_input_tokens} cache_read={u.cache_read_input_tokens} out={u.output_tokens}")
    block = next(b for b in resp.content if b.type == "tool_use")
    return FicheConseillere.model_validate(block.input)

## 3 — Extraire les fiches (seulement les nouvelles)

Lit `transcripts/` sur le Drive → écrit `fiches/<nom>.json` + `.md`. **Ne re-analyse pas** les appels déjà fichés
(gain de temps + tokens). Passe `FORCE = True` pour tout recalculer.

In [ ]:
import glob, os, json
FORCE = False  # True = ré-analyse même les fiches déjà présentes

def fiche_to_md(f):
    L = [f"# Fiche conseillère\n\n**Résumé :** {f.resume_appel}\n", "## Questions de découverte (dans l'ordre)\n"]
    for q in sorted(f.questions_decouverte, key=lambda x: x.ordre):
        ref = f" → `{q.champ_lead}`" if q.champ_lead else ""
        L.append(f"{q.ordre}. **[{q.etape}]** {q.question} — _{q.intention}_{ref}")
    L.append("\n## Données captées → champs de lead\n")
    for d in f.points_donnees:
        L.append(f"- `{d.champ_lead}` = **{d.valeur}** _({d.etape})_ — « {d.verbatim} »")
    L.append("\n## Objections & réponses\n")
    for o in f.objections:
        L.append(f"- **Objection :** {o.objection}\n  - **Réponse ({o.technique}) :** {o.reponse_conseiller}")
    L.append("\n## Phrases d'empathie\n")
    for p in f.phrases_empathie:
        L.append(f"- « {p.phrase} » _({p.contexte})_")
    L.append("\n## Vocabulaire\n- " + "\n- ".join(f.vocabulaire))
    L.append("\n## Transitions\n")
    for t in f.transitions:
        L.append(f"- {t.de} → {t.vers} : « {t.formulation} »")
    L.append("\n## Bonnes pratiques\n- " + "\n- ".join(f.bonnes_pratiques))
    L.append("\n## À éviter\n- " + "\n- ".join(f.a_eviter))
    return "\n".join(L)

nouvelles = 0
for p in sorted(glob.glob(f"{TRANSCRIPTS}/*.txt")):
    nom = os.path.splitext(os.path.basename(p))[0]
    jpath = f"{FICHES}/{nom}.json"
    if os.path.exists(jpath) and not FORCE:
        print(f"⏭️  {nom} — déjà fiché, skip")
        continue
    print(f"📝 {nom} …")
    try:
        f = extraire_fiche(open(p).read())
    except Exception as e:
        print(f"   ⚠️ échec : {e}"); continue
    json.dump(f.model_dump(), open(jpath, "w"), ensure_ascii=False, indent=2)
    open(f"{FICHES}/{nom}.md", "w").write(fiche_to_md(f))
    nouvelles += 1

total = len(glob.glob(f"{FICHES}/*.json"))
print(f"\n✅ {nouvelles} nouvelle(s) fiche(s) · {total} au total dans Drive/fiches")

## 4 — Synthèse : le playbook d'Emma (sur **tout** le corpus)

Relit **toutes** les fiches de `fiches/` (donc tout l'historique accumulé sur le Drive) → régénère
`emma-playbook.md` sur le Drive + le télécharge. Plus tu ajoutes d'appels, plus il s'affine.

In [ ]:
import glob, json

corpus = [json.load(open(p)) for p in sorted(glob.glob(f"{FICHES}/*.json"))]
print(f"📚 {len(corpus)} fiche(s) → synthèse du playbook")

SYNTHESE_SYS = """Tu es responsable formation chez SmartSeniors. À partir de fiches conseillères issues de vrais appels,
tu produis un PLAYBOOK condensé et actionnable pour Emma, l'IA conseillère. Factuel : des règles et des formulations
réutilisables, pas de blabla. Garde les meilleures formulations verbatim."""

prompt = (
    f"Voici {len(corpus)} fiche(s) conseillère(s) en JSON :\n\n"
    + json.dumps(corpus, ensure_ascii=False)
    + "\n\nProduis un playbook **Markdown** prêt à coller dans le prompt système d'Emma, sections :\n"
      "1. Ordre de découverte optimal (une question type par étape, dans le bon ordre)\n"
      "2. Scripts d'objection (objection → réponse → technique)\n"
      "3. Phrases d'empathie réutilisables\n"
      "4. Vocabulaire imposé / à maîtriser\n"
      "5. Carte « ce que la famille dit » → « champ de lead » (étape identite, puis solution)\n"
      "6. Transitions entre étapes\n"
      "7. Pièges à éviter (synthèse des « à éviter »)\n"
      "Condense, dédoublonne, priorise."
)

resp = client.messages.create(model=MODEL, max_tokens=16000, system=SYNTHESE_SYS,
                              messages=[{"role": "user", "content": prompt}])
playbook = "".join(b.text for b in resp.content if b.type == "text")
out = f"{BASE}/emma-playbook.md"
open(out, "w").write(playbook)
print("✅ playbook écrit sur le Drive :", out, "\n")
print(playbook[:3000])

from google.colab import files
files.download(out)

## Et après ?

1. **Ramène le `emma-playbook.md`** (téléchargé ci-dessus, ou direct depuis `MyDrive/SmartSeniors/biblio_emma/`) à Claude Code → il remplace `knowledge/emma-playbook.md` et refait une passe d'enrichissement du `BASE_SYSTEM_PROMPT` (`pages/functions/api/chat.js`).
2. **Enrichir** : dépose simplement d'autres `.m4a` dans `audio/` et relance **§1 → §3 → §4**. Seuls les nouveaux appels sont transcrits/analysés ; le playbook se régénère sur tout le corpus.
3. **RGPD** : ne committe que l'**anonymisé** (`emma-playbook.md`, éventuellement `fiches/`). Les `.m4a` et transcripts bruts restent sur ton Drive privé — jamais dans le repo.